# Tarea 1 · Parte 1: Recolección de noticias de El Comercio

**Pregunta guía:** ¿se pueden convertir titulares peruanos en un clasificador de noticias?

| | |
|---|---|
| Opción elegida | **B · El Comercio** |
| Fecha de descarga | 14 de octubre de 2026 |
| URL base | `https://elcomercio.pe` |
| Fuente de los enlaces | sitemaps de sección `https://elcomercio.pe/sitemap/web/<seccion>/?outputType=xml` |
| Categorías | `politica`, `economia`, `deportes`, `ciencia_tecnologia` |

**Por qué sitemaps y no RSS.** El `robots.txt` de El Comercio solo prohíbe `/catalogo/`, `/buscar/` y parte de `/archivo/` a los bots en general, y declara él mismo sus sitemaps de sección (líneas `Sitemap:`), cada uno con las 100 notas más recientes. El RSS de sección también está permitido, pero el 14 de octubre traía pocas notas en algunas secciones (18 en tecnología, 45 en política), así que no alcanzaba las ~90 por categoría. Del sitemap salen los enlaces; de cada nota se leen sus etiquetas `<meta>` (titular, bajada, fecha y sección).

**Buenas prácticas.** `User-Agent` que identifica el uso académico, pausa de 1,5 segundos entre solicitudes y como máximo un reintento por enlace.

In [1]:
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Tarea academica PUCP - curso Python Intermedio)"}
PAUSA = 1.5          # segundos entre solicitudes
BASE = "https://elcomercio.pe"

RAW = Path("datos/raw")
RAW.mkdir(parents=True, exist_ok=True)

print("Fecha de descarga:", date.today())

Fecha de descarga: 2026-10-14


## 1. Revisar `robots.txt`

Antes de pedir nada, revisamos qué permite el medio.

In [2]:
robots = requests.get(f"{BASE}/robots.txt", headers=HEADERS, timeout=30).text
(RAW / "robots_elcomercio.txt").write_text(robots, encoding="utf-8")

# Reglas para todos los bots (User-agent: *) y los sitemaps de las secciones que usaremos
reglas = robots.split("User-agent: *")[1].split("#")[0]
print(reglas.strip())
for linea in robots.splitlines():
    if linea.startswith("Sitemap:") and "/sitemap/" in linea and any(f"/{s}/" in linea for s in ["politica", "economia", "deporte-total", "tecnologia"]):
        print(linea)

Disallow: /catalogo/
Disallow: /buscar/
Disallow: /archivo/todas/19*
Disallow: /archivo/todas/200*
Sitemap: https://elcomercio.pe/sitemap/politica/?outputType=xml
Sitemap: https://elcomercio.pe/sitemap/tecnologia/?outputType=xml
Sitemap: https://elcomercio.pe/sitemap/economia/?outputType=xml
Sitemap: https://elcomercio.pe/sitemap/deporte-total/?outputType=xml


## 2. Enlaces desde los sitemaps de sección

Una sección por cada categoría de la tarea. El sitemap es un XML con una etiqueta `<loc>` por nota. Guardamos cada XML tal cual en `datos/raw/` como respaldo crudo; si el respaldo ya existe se lee de ahí, para que al volver a ejecutar el notebook se usen los mismos enlaces del día de descarga.

In [3]:
SECCIONES = ["politica", "economia", "deporte-total", "tecnologia"]

def obtener(url):
    """GET con pausa y como máximo un reintento."""
    for intento in range(2):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            time.sleep(PAUSA)
            if r.status_code == 200:
                return r.text
        except requests.RequestException:
            time.sleep(PAUSA)
    return None

enlaces = []
for seccion in SECCIONES:
    archivo = RAW / f"sitemap_{seccion}.xml"
    if not archivo.exists():
        archivo.write_text(obtener(f"{BASE}/sitemap/web/{seccion}/?outputType=xml"), encoding="utf-8")
    xml = archivo.read_text(encoding="utf-8")
    locs = [loc.text.strip() for loc in BeautifulSoup(xml, "xml").find_all("loc")]
    enlaces += [{"seccion_sitemap": seccion, "url": u} for u in locs]
    print(f"{seccion:15} {len(locs)} enlaces")

enlaces = pd.DataFrame(enlaces)
enlaces.shape

politica        100 enlaces
economia        100 enlaces
deporte-total   100 enlaces
tecnologia      100 enlaces


(400, 2)

## 3. Leer cada nota

Probamos primero con una sola nota para ver qué trae. El titular y la bajada vienen en las etiquetas `og:title` y `og:description`, que el medio pone para redes sociales; la fecha en `article:published_time`.

La `seccion_original` sale de la ruta de la URL, tal como la organiza el medio: `elcomercio.pe/economia/mercados/...` → `economia-mercados`.

In [4]:
def seccion_de_url(url):
    partes = url.replace(BASE, "").strip("/").split("/")
    return "-".join(partes[:2]) if len(partes) > 2 else partes[0]

def leer_nota(url):
    html = obtener(url)
    if html is None:
        return {"url": url, "error": "sin respuesta"}
    sopa = BeautifulSoup(html, "html.parser")
    meta = lambda prop: (sopa.find("meta", property=prop) or {}).get("content", "")
    return {
        "titular": meta("og:title").strip(),
        "bajada": meta("og:description").strip(),
        "medio": "El Comercio",
        "seccion_original": seccion_de_url(url),
        "url": url,
        "fecha": meta("article:published_time")[:10],
        "error": "",
    }

leer_nota(enlaces.loc[0, "url"])

{'titular': 'Keiko Fujimori ratifica su respaldo al ministro César Astudillo y al jefe de la PNP Fredy López',
 'bajada': 'Keiko Fujimori ratificó su respaldo al ministro de Defensa, César Astudillo, y al comandante general de la Policía Nacional del Perú (PNP), Fredy López, durante una ceremonia de ascenso de cinco suboficiales reconocidos por su destacada labor en la lucha contra la delincuencia.',
 'medio': 'El Comercio',
 'seccion_original': 'politica-gobierno',
 'url': 'https://elcomercio.pe/politica/gobierno/keiko-fujimori-ratifica-su-respaldo-al-ministro-cesar-astudillo-y-al-jefe-de-la-pnp-ultimas-noticia/',
 'fecha': '2026-10-14',
 'error': ''}

Funciona. Ahora recorremos todos los enlaces. El avance se guarda cada 25 notas en `datos/raw/notas_crudo.csv`; si el proceso se corta (bloqueo, internet), al volver a correr la celda continúa desde donde quedó en vez de pedir todo otra vez.

In [5]:
CRUDO = RAW / "notas_crudo.csv"
crudo = pd.read_csv(CRUDO, keep_default_na=False) if CRUDO.exists() else pd.DataFrame()
hechas = set(crudo["url"]) if len(crudo) else set()
pendientes = [u for u in enlaces["url"] if u not in hechas]
print(f"Ya leídas: {len(hechas)} · pendientes: {len(pendientes)}")

nuevas = []
for i, url in enumerate(pendientes, 1):
    nuevas.append(leer_nota(url))
    if i % 25 == 0 or i == len(pendientes):
        crudo = pd.concat([crudo, pd.DataFrame(nuevas)], ignore_index=True)
        crudo.to_csv(CRUDO, index=False)
        nuevas = []
        print(f"{i}/{len(pendientes)}")

crudo = crudo.merge(enlaces, on="url", how="left")
crudo.shape

Ya leídas: 400 · pendientes: 0


(400, 8)

In [6]:
crudo["error"].replace("", "ok").value_counts()

error
ok    400
Name: count, dtype: int64

In [7]:
crudo.head()

,titular,bajada,medio,seccion_original,url,fecha,error,seccion_sitemap
0,Keiko Fujimori ratifica su respaldo al ministr...,Keiko Fujimori ratificó su respaldo al ministr...,El Comercio,politica-gobierno,https://elcomercio.pe/politica/gobierno/keiko-...,2026-10-14,,politica
1,Juan Sheput: “El Congreso ha apostado por la c...,"El ministro de Trabajo y Promoción del Empleo,...",El Comercio,politica-gobierno,https://elcomercio.pe/politica/gobierno/juan-s...,2026-10-14,,politica
2,"“Grupos hostiles”, el factor DINI y otras acci...",El Senado debatirá el pedido de facultades leg...,El Comercio,politica,https://elcomercio.pe/politica/grupos-hostiles...,2026-10-14,,politica
3,Senadora Milagros Jáuregui a favor de analizar...,En el programa “Tenemos Que Hablar” de El Come...,El Comercio,politica-elecciones,https://elcomercio.pe/politica/elecciones/sena...,2026-10-14,,politica
4,Ley MAPE: Minem prevé presentar propuesta este...,El Minem presentará al Congreso su propuesta d...,El Comercio,politica,https://elcomercio.pe/politica/ley-mape-minem-...,2026-10-14,,politica
